# Spherical mass mapping with MCLMC: the joint posterior of cosmology and initial conditions

Notebook 14 finds the maximum a posteriori (MAP) initial conditions (IC) behind two convergence maps at fixed cosmology. Here we sample the full posterior instead, over $(\Omega_c,\sigma_8)$ **and** the $\mathrm{MESH}^3$ white-noise IC field jointly, with Microcanonical Langevin Monte Carlo (MCLMC, BlackJAX) on one A100.

The forward model and the data are those of notebook 14, Part II (Euclid):
- Planck18 linear IC, 2LPT on the lightcone;
- spherical painting into capped equal-volume shells from `R_MIN`, each low-passed at the multipole the mesh resolves;
- Born convergence for DES Y3 source bins 2 and 3, with the Euclid IST:F source density (7.5 galaxies/arcmin² per bin) and shape noise $\sigma_e=0.30/\sqrt2$ per component;
- a pixel likelihood on $\kappa$ band-limited to $2\le\ell\le$ `ELL_MAX`.

The priors are uniform, $\Omega_c\in[0.1,0.5]$ and $\sigma_8\in[0.6,1.0]$, and the sampler moves their probit-transformed base variables. The chain starts from an independent random draw (`INIT = "random"`); `INIT = "truth"` starts it at the truth instead, the standard check that the sampler holds the posterior once it is there.

We then push posterior samples through the forward model to obtain posterior-predictive $\kappa$ maps, and compare their mean and scatter with the truth: the maps themselves, their coherence and transfer function, and the starlet $\ell_1$ norm. The comparison uses the band-limited $\kappa$, the quantity the likelihood constrains.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q uv
    ![ -d /content/jax-fli ] || git clone -q -b mclmc-mass-mapping https://github.com/ASKabalan/jax-fli.git /content/jax-fli
    !cd /content/jax-fli && git pull -q && git log -1 --oneline
    !cd /content/jax-fli && uv pip install --system -q -e ".[cuda,catalog,sampling,plot,examples,starlet]" arviz-plots

## 1. Imports and environment

The GPU memory is not preallocated, because the CUDA spherical-harmonic transform keeps its cuFFT plans outside XLA's pool. Every spherical-harmonic transform, in the likelihood and in the predictions, uses `jax_cuda`.

In [ ]:
import os

os.environ["JAX_ENABLE_X64"] = "True"  # float32 IC gradients diverge through the LPT + painting chain
os.environ["JAX_PLATFORMS"] = "cuda,cpu"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

import gc
import importlib.metadata as md
import json
import shutil
import time
from pathlib import Path

import datasets

datasets.disable_progress_bar()

import healpy as hp
import jax
import jax.numpy as jnp
import jax_cosmo as jc
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist
import pyarrow.parquet as pq
from blackjax.mcmc.integrators import IntegratorState
from numpyro.handlers import condition, reparam
from numpyro.infer.reparam import Reparam
from numpyro.infer.util import potential_energy
from scipy.special import ndtri

import jax_fli as jfli
from jax_fli.data.nz import get_des_y3_nz_shear, plot_nz
from jax_fli.initial import interpolate_initial_conditions
from jax_fli.summary_statistics import starlet_coefficients_spherical

jax.config.update("jax_enable_x64", True)
MAP2ALM = "jax_cuda"
T_START = time.time()


def log(*args):
    # timestamped, flushed progress line (seconds since this cell)
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - T_START:6.0f}s]", *args, flush=True)


for pkg in ("jax", "jax-fli", "jaxpm", "jaxdecomp", "jax-cosmo", "s2fft", "blackjax"):
    du = json.loads(md.distribution(pkg).read_text("direct_url.json") or "{}").get("vcs_info", {})
    print(f"  {pkg:>9} {md.version(pkg)}  {du.get('requested_revision', '')}@{du.get('commit_id', '')[:10]}")
from s2fft_lib import _s2fft

log(f"devices {jax.devices()}  x64 {jax.config.jax_enable_x64}  s2fft CUDA {_s2fft.COMPILED_WITH_CUDA}")

## 2. Parameters

`MESH` is the resolution; `ELL_MAX`, `PAINT_NSIDE` and `NSIDE` follow from it as in notebook 14. One MCLMC kernel step is one integration step with two gradient evaluations. The tuning runs `NUM_WARMUP` steps, and its last 20 % keeps every position in memory, so `NUM_WARMUP` is bounded by the GPU memory at large meshes. After tuning, each batch stores `NUM_SAMPLES` draws, one every `THINNING` steps: `BURN_BATCHES` batches of burn-in, then `BATCH_COUNT` batches of the posterior (section 6). The next cell holds the values papermill can override.

In [ ]:
MESH = 64
INIT = "random"  # "random": independent draw at INIT_SCALE, cosmology at the prior centre; "truth": start at the truth
NUM_WARMUP = 50
NUM_SAMPLES = 5  # stored draws per batch
BATCH_COUNT = 16  # posterior batches
THINNING = 20  # MCLMC steps per stored draw
BURN_BATCHES = 7  # burn-in batches before the re-tuning (0 skips the burn-in stage)
ENERGY_VAR = 1e-4
INIT_STEP = 0.02  # initial tuning step, as a fraction of sqrt(dimension)
DIAG_PRECOND = False
L_FACTOR = 1.0  # momentum decoherence length L = L_FACTOR * sqrt(dimension), section 6
INIT_SCALE = 0.3
ROOT = "."

In [ ]:
LPT_ORDER = 2
# lightcone: capped equal-volume shells from R_MIN (notebook 14, section 3)
N_SHELLS, MIN_WIDTH, MAX_WIDTH, R_MIN = 22, 50.0, 150.0, 300.0  # Mpc/h
DES_BINS = (1, 2)  # DES Y3 bins 2 and 3
MAX_Z = 1.0
ELL_MIN = 2
SEED = 0

cosmo = jc.Planck18()
box = tuple(float(x) for x in jfli.utils.compute_box_size_from_redshift(cosmo, MAX_Z, (0.5, 0.5, 0.5)))
L, chi_max = box[0], box[0] / 2
n_bins = len(DES_BINS)
# Euclid IST:F: 30 galaxies/arcmin^2 over four DES-shaped bins, total ellipticity dispersion 0.30 -> 0.30 / sqrt 2 per component
NZ = [get_des_y3_nz_shear(gals_per_arcmin2=[30.0 / 4] * 4, zmax=MAX_Z)[i] for i in DES_BINS]
SIGMA_E = 0.30 / np.sqrt(2)
OUT = Path(ROOT) / f"MCLMC_MESH{MESH}_EUCLID_{INIT}"
CHAIN = OUT / "chain"
OUT.mkdir(parents=True, exist_ok=True)
zz = jnp.linspace(1e-3, MAX_Z, 512)
BIN = [
    f"source bin {i + 1} (<z> = {float(jnp.trapezoid(zz * nz(zz), zz) / jnp.trapezoid(nz(zz), zz)):.2f})"
    for i, nz in zip(DES_BINS, NZ)
]

# lensing efficiency q_b(chi) -> the multipole the mesh resolves at the peak of the lower bin
chi_grid = np.linspace(1.0, chi_max, 2000)
z_s = np.linspace(1e-3, MAX_Z, 1000)
chi_s = np.asarray(jc.background.radial_comoving_distance(cosmo, jc.utils.z2a(jnp.asarray(z_s))))
a_grid = np.asarray(jc.background.a_of_chi(cosmo, jnp.asarray(chi_grid)))
q = np.array(
    [
        1.5
        * cosmo.Omega_m
        / 2997.92458**2
        * chi_grid
        / a_grid
        * np.trapezoid(
            np.array(nz(jnp.asarray(z_s)))[None] * np.clip(1 - chi_grid[:, None] / chi_s, 0, None), z_s, axis=1
        )
        / np.trapezoid(np.array(nz(jnp.asarray(z_s))), z_s)
        for nz in NZ
    ]
)
chi_peak = chi_grid[q.argmax(axis=1)]
k_nyq = np.pi * MESH / L
ELL_MAX = int(np.floor(k_nyq * chi_peak.min()))
ELL_TAPER = int(round(0.125 * ELL_MAX))
PAINT_NSIDE = int(2 ** np.ceil(np.log2(ELL_MAX / 2)))
NSIDE = 2 * PAINT_NSIDE
N_GAL = np.array([float(n.gals_per_arcmin2) for n in NZ])
SIGMA_B = SIGMA_E / np.sqrt(N_GAL * hp.nside2pixarea(NSIDE, degrees=True) * 3600)  # noise per pixel and bin
print(f"box L = 2 chi(z={MAX_Z}) = {L:.1f} Mpc/h   cell {L / MESH:.2f} Mpc/h   k_Nyq {k_nyq:.4f} h/Mpc")
print(
    f"lensing-efficiency peaks at chi = {np.round(chi_peak)} Mpc/h -> ELL_MAX = {ELL_MAX}, taper {ELL_TAPER}, "
    f"ELL_MIN = {ELL_MIN}, PAINT_NSIDE = {PAINT_NSIDE}, NSIDE = {NSIDE}"
)
print(
    f"Euclid: {N_GAL.tolist()} galaxies/arcmin^2 per bin, sigma_e {SIGMA_E:.3f} -> noise per pixel {np.round(SIGMA_B, 5)}"
)
print(
    f"MCLMC: start {INIT}, {NUM_WARMUP} tuning steps, {BATCH_COUNT} batches x {NUM_SAMPLES} draws x {THINNING} steps "
    f"({BURN_BATCHES} batches of burn-in), output {OUT}"
)
plot_nz(NZ, cosmo=cosmo, labels=[f"DES Y3 bin {i + 1}" for i in DES_BINS])
plt.show()

## 3. Model

The configuration is notebook 14's `make_config`: `full_field_probmodel` in LPT mode with capped equal-volume shells, the inner edge `r_min`, the per-shell resolution cut and $\ell\ge2$. The cosmology priors are left free, so the model has $2+\mathrm{MESH}^3$ parameters.

In [ ]:
priors = {"Omega_c": jfli.infer.PreconditionnedUniform(0.1, 0.5), "sigma8": jfli.infer.PreconditionnedUniform(0.6, 1.0)}

config = jfli.ppl.Configurations(
    mesh_size=(MESH,) * 3,
    box_size=box,
    halo_size=(0, 0),
    field_sharding=None,
    sim_mode="lpt",
    lpt_order=LPT_ORDER,
    paint_order="cic",
    gradient_order=4,
    laplace_fd=True,
    number_of_shells=N_SHELLS,
    shell_spacing="equal_vol",
    min_width=MIN_WIDTH,
    max_width=MAX_WIDTH,
    r_min=R_MIN,
    resolution_cut=True,
    geometry="spherical",
    scheme="bilinear",
    nside=NSIDE,
    paint_nside=PAINT_NSIDE,
    kernel_width_pixels=0.8,
    observer_position=(0.5, 0.5, 0.5),
    lensing_output="convergence",
    normalization="global",
    map2alm_method=MAP2ALM,
    min_redshift=0.001,
    max_redshift=MAX_Z,
    n_integrate=8,
    fiducial_cosmology=jc.Planck18,
    nz_shear=NZ,
    priors=priors,
    sigma_e=SIGMA_E,
    ell_max=ELL_MAX,
    ell_taper_width=ELL_TAPER,
    ell_min=ELL_MIN,
)
model, forward = jfli.ppl.full_field_probmodel(config), jfli.ppl.make_full_field_model(config)

_, r_c, widths = jfli.resolve_geometry(
    cosmo, chi_max, nb_shells=N_SHELLS, shell_spacing="equal_vol", min_width=MIN_WIDTH, max_width=MAX_WIDTH, r_min=R_MIN
)
order = np.argsort(np.asarray(r_c))
r_lo = np.asarray(r_c)[order] - np.asarray(widths)[order] / 2
r_hi = np.asarray(r_c)[order] + np.asarray(widths)[order] / 2
print("shell edges [Mpc/h]: " + " ".join(f"{x:.0f}" for x in np.append(r_lo, r_hi[-1])))
print(f"sampled dimension: 2 + {MESH}^3 = {2 + MESH**3:,}")

## 4. Truth and data

The truth is Planck18 with the white-noise draw at `SEED`, as in notebook 14. `colour` turns a white field into the physical IC at a given cosmology, and `band_kappa` runs the forward model and band-limits both $\kappa$ maps to $2\le\ell\le$ `ELL_MAX`, the quantity the likelihood compares. The data are the band-limited truth plus white noise of per-pixel standard deviation $\sigma_b=\sigma_e/\sqrt{n_b\,\Omega_{\rm pix}}$. The truth IC, the truth $\kappa$ and the data go to parquet.

In [ ]:
def colour(white, cosmo_):
    return interpolate_initial_conditions(
        white,
        config.mesh_size,
        config.box_size,
        cosmo=cosmo_,
        observer_position=config.observer_position,
        halo_size=config.halo_size,
        nside=config.nside,
    )


@jax.jit
def band_kappa(ic, cosmo_):
    kappa = forward(cosmo_, ic)[0]
    band = [kappa[b].scale_cut(ELL_MAX, ELL_TAPER, l_min=ELL_MIN, method=MAP2ALM).array for b in range(n_bins)]
    return kappa.replace(array=jnp.stack(band))


zeros = jnp.zeros(MESH)
sig = jnp.asarray(SIGMA_B)[:, None]
w_truth = jax.random.normal(jax.random.PRNGKey(SEED), (MESH,) * 3)
eps = jax.random.normal(jax.random.PRNGKey(SEED + 1000), (n_bins, 12 * NSIDE**2))

log("truth forward (compiles band_kappa) ...")
ic_truth = colour(w_truth, cosmo).replace(
    name="true_ic", z_sources=zeros, comoving_centers=zeros, scale_factors=zeros, density_width=zeros
)
kappa_truth = band_kappa(ic_truth, cosmo)
x_obs = kappa_truth.array + sig * eps
truth_cat = jfli.io.Catalog(field=[ic_truth], cosmology=[cosmo])
truth_cat.to_parquet(str(OUT / "true_ic.parquet"))
jfli.io.Catalog(field=[kappa_truth], cosmology=[cosmo]).to_parquet(str(OUT / "truth_kappa.parquet"))
jfli.io.Catalog(field=[kappa_truth.replace(array=x_obs)], cosmology=[cosmo]).to_parquet(
    str(OUT / "observed_kappa.parquet")
)
log("truth written")
for b in range(n_bins):
    print(f"  {BIN[b]}: kappa rms {float(kappa_truth.array[b].std()):.2e}   noise {float(sig[b, 0]):.2e} per pixel")
fig, axes = plt.subplots(n_bins, 2, figsize=(10, 3.2 * n_bins), squeeze=False)
kappa_truth.plot(ax=list(axes[:, 0]), titles=[f"truth $\\kappa$, {z}" for z in BIN])
kappa_truth.replace(array=x_obs).plot(ax=list(axes[:, 1]), titles=[f"data = truth + shape noise, {z}" for z in BIN])
plt.show()

## 5. The potential and the cosmology preconditioner

MCLMC samples $\log p(w,\theta\mid d)$ over the white field $w$ and the base variables $\theta$ of the two cosmological parameters, with the data sites conditioned. Before sampling we measure the compiled memory and the time of one value-and-gradient call, and compare the gradient with a central finite difference along a random direction that moves both the field and the cosmology.

The two base variables are far narrower than the field. On the 64³ mock, a power iteration on Hessian-vector products at the truth gives a largest eigenvalue of about $3\times10^3$, 96 % of it along `Omega_c_base`, i.e. a posterior width of 0.018 against the unit prior width of the field. MCLMC with a unit mass matrix then needs a step size of order that width times $\sqrt d$, and the tuner returned $\varepsilon\approx0.6$ for $\sqrt d=512$. We therefore sample $z_k=\theta_k/s_k$, with $s_k=(\partial^2U/\partial\theta_k^2)^{-1/2}$ the conditional width of each base variable at the fiducial point, measured by a central difference of the compiled gradient. A NumPyro `reparam` handler around `full_field_probmodel` does the change of variables; the prior is unchanged, and $\theta_k$, the cosmology and the observables are computed from $z_k$ inside the model.

In [ ]:
class ScaleReparam(Reparam):
    # theta = scale * z: the sampler moves z, of posterior width ~1 when scale is the posterior width of theta
    def __init__(self, scale):
        self.scale = scale

    def __call__(self, name, fn, obs):
        affine = dist.transforms.AffineTransform(0.0, 1.0 / self.scale)
        z = numpyro.sample(f"{name}_z", dist.TransformedDistribution(fn, affine))
        return None, self.scale * z


base_fid = {
    f"{k}_base": float(ndtri((float(getattr(cosmo, k)) - float(p.low)) / (float(p.high) - float(p.low))))
    for k, p in priors.items()
}
data = {f"observable_{i}": x_obs[i] for i in range(n_bins)}
truth_base = {k: jnp.asarray(v) for k, v in base_fid.items()} | {"initial_conditions": w_truth}
potential = jax.jit(lambda pos: potential_energy(condition(model, data=data), (), {}, pos))
vg = jax.jit(jax.value_and_grad(potential))
log("compiling value_and_grad ...")
vg_compiled = vg.lower(truth_base).compile()
log(f"XLA temporaries {vg_compiled.memory_analysis().temp_size_in_bytes / 1e9:.1f} GB")
u_truth, g_truth = jax.block_until_ready(vg_compiled(truth_base))
t0 = time.time()
jax.block_until_ready(vg_compiled(truth_base))
log(f"value_and_grad {time.time() - t0:.2f} s per call")
keys = jax.random.split(jax.random.PRNGKey(32), 3)
v = {k: jax.random.normal(kk, jnp.shape(x)) for (k, x), kk in zip(truth_base.items(), keys)}
nv = jnp.sqrt(sum(jnp.sum(x**2) for x in v.values()))
v = jax.tree.map(lambda x: x / nv, v)
shift = lambda s: jax.tree.map(lambda p, d: p + s * d, truth_base, v)
fd = (float(potential(shift(1e-4))) - float(potential(shift(-1e-4)))) / 2e-4
ad = float(sum(jnp.vdot(g_truth[k], v[k]) for k in v))
log(f"FD check: autodiff {ad:.8e}  finite difference {fd:.8e}")
chi2_truth = float(jnp.sum(((x_obs - kappa_truth.array) / sig) ** 2)) / x_obs.size
print(f"at the truth: potential {float(u_truth):.6e}   chi2/N_pix {chi2_truth:.4f}")

# conditional width of each base variable at the fiducial point: central difference of the compiled gradient
H = 1e-3
PRECOND = {}
for k in base_fid:
    g_plus = vg_compiled(truth_base | {k: truth_base[k] + H})[1][k]
    g_minus = vg_compiled(truth_base | {k: truth_base[k] - H})[1][k]
    PRECOND[k] = float(1.0 / np.sqrt((g_plus - g_minus) / (2 * H)))
print("conditional widths of the base variables:", {k: round(x, 5) for k, x in PRECOND.items()})
cond_model = condition(reparam(model, config={k: ScaleReparam(s_k) for k, s_k in PRECOND.items()}), data=data)
truth_position = {f"{k}_z": jnp.asarray(base_fid[k] / s_k) for k, s_k in PRECOND.items()} | {
    "initial_conditions": w_truth
}
random_position = {f"{k}_z": jnp.asarray(0.0) for k in PRECOND} | {
    "initial_conditions": INIT_SCALE * jax.random.normal(jax.random.PRNGKey(30), (MESH,) * 3)
}
init_position = truth_position if INIT == "truth" else random_position
print(
    f"start: {INIT}, cosmology z = { ({k: round(float(x), 3) for k, x in init_position.items() if k != 'initial_conditions'}) }"
)

## 6. MCLMC

`jfli.infer.batched_sampling` tunes the step size and the momentum decoherence length $L$, then samples in batches and saves the sampler state after each one, so re-running this cell resumes an interrupted chain. MCLMC is unadjusted: its bias is controlled by the energy error per dimension, `ENERGY_VAR`, and `metrics.md` reports the fraction of NaN-free steps.

The tuning starts from a step of `INIT_STEP` $\sqrt d$. We keep the tuned step size but set $L$ ourselves. BlackJAX estimates $L$ from the effective sample size of the last 20 % of the tuning steps, which bounds it by roughly $0.4\,\varepsilon$ times that number of steps. At our dimension the chain decorrelates only after $\sim\sqrt d/\varepsilon$ steps, so a tuning run short enough to fit in memory returns an $L$ of order the step size, and the chain diffuses instead of moving ballistically: in the 64³ smoke test the tuner gave $L=1.2$ and the cosmology did not move over 75 steps. The white field has unit prior variance, and the likelihood constrains only its large-scale modes, so the sum of the posterior variances is close to $d$ and we set $L=$ `L_FACTOR` $\sqrt d$, the variance-based estimate BlackJAX itself uses after its second phase. A first `batched_sampling` call with `batch_count=0` tunes and writes the sampler state; we replace $L$ in that state (and keep a unit mass matrix), and the second call samples from it.

The step size is tuned where the chain starts. From a random start that point lies far from the typical set, and on the 64³ mock the tuner returned half the step size it finds at the truth. The chain therefore runs in two stages, each a `run_stage`: `BURN_BATCHES` batches of burn-in from the random start, then a fresh tuning from the last burn-in position and the `BATCH_COUNT` batches of the posterior, in `chain/burnin/` and `chain/posterior/`. Each stored draw is recoloured to the physical IC with its own cosmology (`jfli.infer.colour_ic`) and cast to float32 before `jfli.infer.sample2catalog` writes it to parquet.

In [ ]:
colour_sample = jfli.infer.colour_ic(config)


def post_process(sample):
    sample = colour_sample(sample)
    return sample | {"initial_conditions": sample["initial_conditions"].astype(jnp.float32)}


del g_truth, vg_compiled
gc.collect()
dim = sum(int(np.size(x)) for x in jax.tree.leaves(init_position))
sampling = dict(
    path=str(CHAIN),
    rng_key=jax.random.PRNGKey(1),
    num_warmup=NUM_WARMUP,
    num_samples=NUM_SAMPLES,
    sampler="MCLMC",
    thinning=THINNING,
    init_params=init_position,
    progress_bar=True,
    print_rate=max(1, NUM_WARMUP // 20),
    save_callback=jfli.infer.sample2catalog(config),
    post_process=post_process,
    mclmc_desired_energy_var=ENERGY_VAR,
    mclmc_init_step_size=INIT_STEP * np.sqrt(dim),
    mclmc_diagonal_preconditioning=DIAG_PRECOND,
)
f64 = lambda x: jax.ShapeDtypeStruct(jnp.shape(x), jnp.float64)
pos = {k: f64(x) for k, x in init_position.items()}
abstract = {  # mirrors batched_sampling's saved MCLMC state
    "nb_samples": jax.ShapeDtypeStruct((), jnp.int64),
    "last_state": IntegratorState(pos, pos, f64(0.0), pos),
    "parameters": {
        "L": f64(0.0),
        "step_size": f64(0.0),
        "inverse_mass_matrix": jax.ShapeDtypeStruct((dim,), jnp.float64),
    },
}


def run_stage(path, start, batch_count, key):
    # tune from `start`, replace L by L_FACTOR sqrt(d), sample `batch_count` batches; returns the last position
    state_path = path / "sampling_state"
    kw = sampling | dict(path=str(path), rng_key=key, init_params=start)
    if not state_path.exists():
        t0 = time.time()
        jfli.infer.batched_sampling(cond_model, batch_count=0, **kw)  # tuning only: writes the state and returns
        state = jfli.io.load_sharded(str(state_path), abstract_pytree=abstract)
        tuned = {k: float(state["parameters"][k]) for k in ("L", "step_size")}
        state["parameters"] = state["parameters"] | {
            "L": jnp.asarray(L_FACTOR * np.sqrt(dim)),
            "inverse_mass_matrix": jnp.ones(dim),
        }
        jfli.io.save_sharded(state, str(state_path), overwrite=True, dump_structure=False)
        (path / "L_override.json").write_text(json.dumps({"tuned": tuned, "L": L_FACTOR * np.sqrt(dim), "dim": dim}))
        log(
            f"{path.name}: tuned in {time.time() - t0:.0f} s: L {tuned['L']:.3g}, step size {tuned['step_size']:.4g} -> L = {L_FACTOR * np.sqrt(dim):.4g}"
        )
        del state
    t0 = time.time()
    jfli.infer.batched_sampling(cond_model, batch_count=batch_count, **kw)
    log(f"{path.name}: {batch_count} batches done ({time.time() - t0:.0f} s this call)")
    print((path / "metrics.md").read_text())
    last = jfli.io.load_sharded(str(state_path), abstract_pytree=abstract)["last_state"].position
    return {k: jnp.asarray(x) for k, x in last.items()}


BURN, POST = CHAIN / "burnin", CHAIN / "posterior"
start = init_position
if BURN_BATCHES > 0:
    start = run_stage(BURN, init_position, BURN_BATCHES, jax.random.PRNGKey(1))
run_stage(POST, start, BATCH_COUNT, jax.random.PRNGKey(2))
log("MCLMC done")

## 7. The chain

The trace of both cosmological parameters over every stored draw, burn-in included (shaded), against the truth. The posterior below reads only the draws of the second stage.

In [ ]:
sample_dir = POST / "samples" / "samples"
batch_files = lambda d: sorted(d.glob("samples_*.parquet"), key=lambda p: int(p.stem.split("_")[-1]))
burn_files, post_files = batch_files(BURN / "samples" / "samples"), batch_files(sample_dir)
trace = {
    k: np.concatenate([pq.read_table(f, columns=[k])[k].to_numpy() for f in burn_files + post_files]) for k in priors
}
n_burn = len(burn_files) * NUM_SAMPLES
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
for ax, k in zip(axes, priors):
    ax.plot(trace[k], lw=0.9)
    ax.axhline(float(getattr(cosmo, k)), color="k", ls="--", lw=1, label="truth")
    if n_burn:
        ax.axvspan(0, n_burn - 0.5, color="0.9", label="burn-in")
    ax.set(xlabel=f"stored draw (one every {THINNING} steps)", ylabel=k)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"burn-in draws {n_burn}, posterior draws {len(post_files) * NUM_SAMPLES}")

## 8. Posterior of the cosmology and the IC

`jfli.io.extract_catalog` streams the kept draws and accumulates the cosmology, the mean and standard deviation of the physical IC, and the IC transfer function and coherence against the truth. `plot_posterior` draws the marginals of $(\Omega_c,\sigma_8)$ with the truth, and `analyze` the IC panels, the IC spectra, the rank and trace plots and the ESS / $\hat R$ table.

In [ ]:
post = jfli.io.extract_catalog(
    set_name=f"MCLMC, {INIT} start",
    cosmo_keys=["Omega_c", "sigma8"],
    patterns=[str(POST / "samples")],
    truth=truth_cat,
    field_statistic=True,
    power_statistic=True,
)
for k in priors:
    m, sd, t = float(post.cosmo[k].mean()), float(post.cosmo[k].std()), post.truth_cosmo[k]
    print(f"{k:8s} {m:.4f} +/- {sd:.4f}   truth {t:.4f}   ({(m - t) / sd:+.2f} sigma)")
jfli.infer.plot_posterior([post], labels={"Omega_c": r"\Omega_c", "sigma8": r"\sigma_8"})
plt.show()

In [ ]:
jfli.infer.analyze(post)

## 9. Posterior-predictive $\kappa$

Every kept draw, its physical IC and its cosmology, goes through the forward model and the same band limit as the likelihood, and the predicted $\kappa$ pair is written to `pred_kappa/`. For each draw we also compute, per source bin, the coherence $r(\ell)=C_\ell^{tp}/\sqrt{C_\ell^{tt}C_\ell^{pp}}$ and the transfer function $T(\ell)=\sqrt{C_\ell^{pp}/C_\ell^{tt}}$ against the truth $t$, and the starlet decomposition of section 11.

In [ ]:
ST_NSIDE, N_SCALES = int(2 ** np.ceil(np.log2(ELL_MAX / 3))), 5
NU_EDGES = np.linspace(-5.0, 5.0, 41)
NU = 0.5 * (NU_EDGES[1:] + NU_EDGES[:-1])
ell = np.arange(ELL_MAX + 1)
sel_l = (ell >= ELL_MIN) & (ell < ELL_MAX)  # the taper weight is 0 at ELL_MAX
resample = lambda m: hp.alm2map(hp.map2alm(np.asarray(m, dtype=np.float64), lmax=ELL_MAX), ST_NSIDE, lmax=ELL_MAX)
starlet = lambda m: starlet_coefficients_spherical(resample(m), nside=ST_NSIDE, nscales=N_SCALES)[0]


def l1_norm(coef, sigma):
    # per scale: nu = w_j / sigma_j, then the sum of |nu| over the pixels of each nu bin (Ajani et al. 2021)
    out = np.zeros((N_SCALES, len(NU)))
    for j in range(N_SCALES):
        nu = coef[j] / sigma[j]
        idx = np.digitize(nu, NU_EDGES) - 1
        ok = (idx >= 0) & (idx < len(NU))
        out[j] = np.bincount(idx[ok], weights=np.abs(nu[ok]), minlength=len(NU))
    return out


kt = np.asarray(kappa_truth.array)
c_tt = np.array([hp.anafast(kt[b], lmax=ELL_MAX) for b in range(n_bins)])
st_truth = [starlet(kt[b]) for b in range(n_bins)]
st_sigma = [s.std(axis=1) for s in st_truth]
l1_truth = np.array([l1_norm(st_truth[b], st_sigma[b]) for b in range(n_bins)])  # (bin, scale, nu)

pred_dir = OUT / "pred_kappa"
shutil.rmtree(pred_dir, ignore_errors=True)
pred_dir.mkdir()
gc.collect()
jax.clear_caches()
stream = datasets.load_dataset(
    "parquet", data_files=str(sample_dir / "*.parquet"), split="train", streaming=True
).with_format("numpy")
coh, trans, l1_pred, pred_cosmo = [], [], [], []
t0 = time.time()
for n, row in enumerate(stream):
    cat = jfli.io.Catalog.from_dataset(row)
    c = cat.cosmology[0]
    cosmo_s = jc.Planck18(Omega_c=jnp.asarray(float(c.Omega_c)), sigma8=jnp.asarray(float(c.sigma8)))
    ic_s = ic_truth.replace(array=jnp.asarray(cat.field[0].array, dtype=jnp.float64))
    k_s = band_kappa(ic_s, cosmo_s)
    jfli.io.Catalog(field=[k_s.replace(name=f"pred_kappa_{n}")], cosmology=[cosmo_s]).to_parquet(
        str(pred_dir / f"kappa_{n:04d}.parquet")
    )
    kp = np.asarray(k_s.array)
    c_pp = np.array([hp.anafast(kp[b], lmax=ELL_MAX) for b in range(n_bins)])
    c_tp = np.array([hp.anafast(kt[b], kp[b], lmax=ELL_MAX) for b in range(n_bins)])
    coh.append((c_tp / np.sqrt(c_tt * c_pp))[:, sel_l])
    trans.append(np.sqrt(c_pp / c_tt)[:, sel_l])
    l1_pred.append([l1_norm(starlet(kp[b]), st_sigma[b]) for b in range(n_bins)])
    pred_cosmo.append((float(c.Omega_c), float(c.sigma8)))
    if n % 10 == 0:
        log(
            f"prediction {n}: Omega_c {pred_cosmo[-1][0]:.4f} sigma8 {pred_cosmo[-1][1]:.4f}  ({time.time() - t0:.0f} s)"
        )
coh, trans, l1_pred = np.array(coh), np.array(trans), np.array(l1_pred)  # (draw, bin, ell) and (draw, bin, scale, nu)
np.savez(
    OUT / "pred_summaries.npz",
    ell=ell[sel_l],
    coherence=coh,
    transfer=trans,
    l1=l1_pred,
    l1_truth=l1_truth,
    cosmo=np.array(pred_cosmo),
)
log(f"{len(coh)} posterior-predictive kappa pairs in {pred_dir}")

## 10. $\kappa$: truth, posterior mean, posterior standard deviation and residual

`extract_catalog` streams the predicted maps and returns their per-pixel mean and standard deviation. Truth and mean share one colour scale; the residual, truth minus mean, is shown on a symmetric scale.

In [ ]:
kpost = jfli.io.extract_catalog(
    set_name="posterior-predictive kappa",
    cosmo_keys=["Omega_c", "sigma8"],
    patterns=[str(pred_dir / "*.parquet")],
    field_statistic=True,
)
k_mean = kappa_truth.replace(array=jnp.asarray(kpost.mean_field.array[0]))
k_std = kappa_truth.replace(array=jnp.asarray(kpost.std_field.array[0]))
k_diff = kappa_truth.replace(array=kappa_truth.array - k_mean.array)
vals = np.concatenate([kt.ravel(), np.asarray(k_mean.array).ravel()])
lo, hi = np.percentile(vals, [0.1, 99.9])
dm = float(np.percentile(np.abs(np.asarray(k_diff.array)), 99.9))
fig, axes = plt.subplots(n_bins, 4, figsize=(17, 3.4 * n_bins), squeeze=False)
kappa_truth.plot(ax=list(axes[:, 0]), titles=[f"truth $\\kappa$\n{z}" for z in BIN], vmin=lo, vmax=hi)
k_mean.plot(ax=list(axes[:, 1]), titles=[f"posterior mean\n{z}" for z in BIN], vmin=lo, vmax=hi)
k_std.plot(ax=list(axes[:, 2]), titles=[f"posterior std\n{z}" for z in BIN], cmap="viridis")
k_diff.plot(ax=list(axes[:, 3]), titles=[f"truth $-$ mean\n{z}" for z in BIN], vmin=-dm, vmax=dm, cmap="RdBu_r")
plt.show()
chi2_mean = float(jnp.sum(((x_obs - k_mean.array) / sig) ** 2)) / x_obs.size
for b in range(n_bins):
    r = np.corrcoef(kt[b], np.asarray(k_mean.array[b]))[0, 1]
    print(
        f"{BIN[b]}: corr(truth, mean) {r:.3f}   RMS truth {kt[b].std():.2e}  mean {float(k_mean.array[b].std()):.2e}"
        f"  std {float(k_std.array[b].mean()):.2e}  residual {float(k_diff.array[b].std()):.2e}"
    )
print(f"chi2/N_pix: posterior mean {chi2_mean:.4f}, truth {chi2_truth:.4f}")

## 11. Coherence and transfer function of the predicted $\kappa$

The line is the mean over the posterior draws and the band its standard deviation. For Gaussian fields under white noise, the best achievable coherence is that of the Wiener filter of both bins jointly (dotted): with the $2\times2$ matrix $C_\ell$ of the band-limited truth and the noise $N_\ell=\sigma_e^2/\bar n_b$ on the diagonal, the coherence of bin $a$ is $r_a^2=[C(C+N)^{-1}C]_{aa}/C_{aa}$. Posterior draws are not Wiener estimates: a draw carries the posterior scatter on top of the mean, so its coherence lies below the bound even when the posterior is correct, while its transfer function should stay close to 1.

In [ ]:
C = np.array([[hp.anafast(kt[a], kt[b], lmax=ELL_MAX) for b in range(n_bins)] for a in range(n_bins)]).transpose(
    2, 0, 1
)
N_ell = SIGMA_E**2 / (N_GAL * (180 * 60 / np.pi) ** 2)
M = np.einsum("lab,lbc,lcd->lad", C[sel_l], np.linalg.inv(C[sel_l] + np.diag(N_ell)[None]), C[sel_l])
wiener = np.array([np.sqrt(M[:, b, b] / C[sel_l, b, b]) for b in range(n_bins)])
ell_s = ell[sel_l]
for stat, name, ylabel in (
    (coh, "coherence", "coherence $r(\\ell)$ with the truth"),
    (trans, "transfer", "transfer $T(\\ell)=\\sqrt{C_\\ell^{\\rm pred}/C_\\ell^{\\rm truth}}$"),
):
    fig, axes = plt.subplots(1, n_bins, figsize=(6.5 * n_bins, 3.8), squeeze=False)
    for b, ax in enumerate(axes[0]):
        m, s = stat[:, b].mean(axis=0), stat[:, b].std(axis=0)
        ax.plot(ell_s, m, color="C3", lw=1.2, label="posterior mean")
        ax.fill_between(ell_s, m - s, m + s, color="C3", alpha=0.3, lw=0, label="$\\pm1\\sigma$ over draws")
        if name == "coherence":
            ax.plot(ell_s, wiener[b], color="k", ls=":", lw=1.5, label="best possible (joint Wiener)")
        ax.axhline(1.0, color="0.5", ls="--", lw=1)
        ax.set(xlabel="multipole $\\ell$", ylabel=ylabel, title=BIN[b])
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
keep = ell_s <= ELL_MAX - ELL_TAPER
for b in range(n_bins):
    print(
        f"{BIN[b]} (ell {ELL_MIN}-{ELL_MAX - ELL_TAPER}): coherence {coh[:, b, keep].mean():.3f} +/- "
        f"{coh[:, b, keep].mean(axis=1).std():.3f} (Wiener {wiener[b, keep].mean():.3f})   transfer "
        f"{trans[:, b, keep].mean():.3f} +/- {trans[:, b, keep].mean(axis=1).std():.3f}"
    )

## 12. Starlet $\ell_1$ norm of the predicted $\kappa$

The starlet (isotropic undecimated wavelet on the sphere) splits each band-limited map, resampled exactly through its harmonic coefficients onto the smallest `ST_NSIDE` with $3N_{\rm side}\ge$ `ELL_MAX`, into five scales. At each scale $j$ the coefficients are expressed as a signal to noise $\nu=w_j/\sigma_j$, with $\sigma_j$ the standard deviation of the truth's coefficients, and the $\ell_1$ norm of a $\nu$ bin is the sum of $|\nu|$ over its pixels (Ajani et al. 2021). The truth is black and dashed; the posterior mean over draws is the line, its standard deviation the band. Under each norm, mean / truth $-$ 1 where the truth's norm is above 1 % of its peak, with the grey band at $\pm10\%$.

In [ ]:
l_hi = 3 * ST_NSIDE / 2 ** np.arange(N_SCALES)
SCALE_BANDS = [f"$\\ell\\approx${l_hi[j] / 2:.0f}–{l_hi[j]:.0f}" for j in range(N_SCALES - 1)] + [
    f"coarse, $\\ell\\lesssim${l_hi[-1]:.0f}"
]
fig, axes = plt.subplots(
    2 * n_bins,
    N_SCALES,
    figsize=(3.3 * N_SCALES, 2.2 * 2 * n_bins),
    sharex=True,
    gridspec_kw=dict(height_ratios=[2, 1] * n_bins),
    squeeze=False,
)
for b in range(n_bins):
    for j in range(N_SCALES):
        top, bottom = axes[2 * b, j], axes[2 * b + 1, j]
        t = l1_truth[b, j]
        m, s = l1_pred[:, b, j].mean(axis=0), l1_pred[:, b, j].std(axis=0)
        top.plot(NU, t, "k--", lw=1.4, label="truth")
        top.plot(NU, m, color="C3", lw=1.4, label="posterior mean")
        top.fill_between(NU, m - s, m + s, color="C3", alpha=0.3, lw=0, label="$\\pm1\\sigma$ over draws")
        ok = t > 0.01 * t.max()
        bottom.axhspan(-0.1, 0.1, color="0.9")
        bottom.axhline(0.0, color="0.5", lw=0.8)
        bottom.plot(NU[ok], m[ok] / t[ok] - 1, color="C3", lw=1.2)
        bottom.fill_between(NU[ok], (m - s)[ok] / t[ok] - 1, (m + s)[ok] / t[ok] - 1, color="C3", alpha=0.3, lw=0)
        bottom.set_ylim(-0.6, 0.6)
        if b == 0:
            top.set_title(f"scale {j + 1}: {SCALE_BANDS[j]}", fontsize=10)
        for ax in (top, bottom):
            ax.grid(alpha=0.3)
    axes[2 * b, 0].set_ylabel(f"$\\ell_1$ norm\n{BIN[b]}")
    axes[2 * b + 1, 0].set_ylabel("mean / truth $-$ 1")
for ax in axes[-1]:
    ax.set_xlabel("$\\nu = w_j/\\sigma_j^{\\rm truth}$")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02), fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## 13. Numbers

In [ ]:
summary = dict(
    mesh=MESH,
    nside=NSIDE,
    ell_max=ELL_MAX,
    init=INIT,
    num_warmup=NUM_WARMUP,
    draws_kept=int(len(coh)),
    thinning=THINNING,
    sampler=json.loads((POST / "L_override.json").read_text()),
    cosmology_scales=PRECOND,
    cosmo={
        k: dict(mean=float(post.cosmo[k].mean()), std=float(post.cosmo[k].std()), truth=post.truth_cosmo[k])
        for k in priors
    },
    chi2_truth=chi2_truth,
    chi2_posterior_mean=chi2_mean,
    kappa_coherence=[float(coh[:, b, keep].mean()) for b in range(n_bins)],
    kappa_wiener=[float(wiener[b, keep].mean()) for b in range(n_bins)],
    kappa_transfer=[float(trans[:, b, keep].mean()) for b in range(n_bins)],
    l1_ratio=[(l1_pred[:, b].mean(axis=0).sum(axis=1) / l1_truth[b].sum(axis=1)).tolist() for b in range(n_bins)],
    minutes=(time.time() - T_START) / 60,
)
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print((POST / "metrics.md").read_text())